# Exercise 1 - SVR (Support Vector Regressor) on the housing dataset

In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np

from sklearn.svm import SVR
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import RandomizedSearchCV, train_test_split, GridSearchCV
from scipy.stats import expon, reciprocal, randint, uniform
from io import BytesIO
from urllib.request import urlopen
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.neighbors import KNeighborsRegressor

In [2]:
url = "https://raw.githubusercontent.com/ageron/handson-ml/master/datasets/housing/housing.csv"
housing = pd.read_csv(BytesIO(urlopen(url).read()))
housing.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [3]:
train_set, test_set = train_test_split(housing, test_size=0.2, random_state=42)
X_train = train_set.drop("median_house_value", axis=1)
y_train = train_set["median_house_value"].copy()
num_attribs = X_train.select_dtypes(include=np.number).columns
cat_attribs = ["ocean_proximity"]
num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
preprocessing = ColumnTransformer([("num", num_pipeline, num_attribs), ("cat", OneHotEncoder(handle_unknown="ignore"), cat_attribs)])

In [4]:
X_train_small, y_train_small = X_train[:5000], y_train[:5000]
svr_pipeline = Pipeline([("preprocessing", preprocessing), ("svr", SVR())])

param_grid = [{"svr__kernel": ["linear"], "svr__C": [0.1, 1, 10, 100]},
    {"svr__kernel": ["rbf"], "svr__C": [0.1, 1, 10, 100], "svr__gamma": [0.01, 0.1, 1, 10]}]

svr_search = GridSearchCV(svr_pipeline, param_grid, cv=3, scoring="neg_root_mean_squared_error", n_jobs=-1)
svr_search.fit(X_train_small, y_train_small)

print("Best parameters:", svr_search.best_params_)
print("Best CV RMSE:", -svr_search.best_score_)

Best parameters: {'svr__C': 100, 'svr__kernel': 'linear'}
Best CV RMSE: 76252.24370398902


# Exercise 2 - RandomisedSearchCV on SVR

In [5]:
param_distribs = [{"svr__kernel": ["linear"], "svr__C": expon(scale=10)},
    {"svr__kernel": ["rbf"], "svr__C": reciprocal(0.1, 100), "svr__gamma": expon(scale=0.1)}]

svr_random = RandomizedSearchCV(svr_pipeline, param_distributions=param_distribs, n_iter=30, cv=3, scoring="neg_root_mean_squared_error",
                                n_jobs=-1, random_state=42)
svr_random.fit(X_train_small, y_train_small)

print("Best parameters:", svr_random.best_params_)
print("Best CV RMSE:", -svr_random.best_score_)

Best parameters: {'svr__C': np.float64(33.706303424611725), 'svr__kernel': 'linear'}
Best CV RMSE: 87046.88800639358


# Exercise 3 - SelectFromModel transformer on SVR

In [6]:
selector = SelectFromModel(RandomForestRegressor(n_estimators=100, random_state=42), threshold="median")
svr_selected = make_pipeline(preprocessing, selector, SVR())

grid_selected = GridSearchCV(svr_selected, param_grid, cv=3, scoring="neg_root_mean_squared_error", n_jobs=-1)
grid_selected.fit(X_train_small, y_train_small)

print("Best parameters:", grid_selected.best_params_)
print("Best CV RMSE:", -grid_selected.best_score_)

Best parameters: {'svr__C': 100, 'svr__kernel': 'linear'}
Best CV RMSE: 77596.10608639423


- SelectFromModel is a transformer which selects import features from the dataset based on the weights of a previously trained estimator (SVR).
- Here, the randomised search performs better than the RandomisedSeach but worse than the GridSearch possibly because it could not find the optimal one among the sampled ones that Grid search could find.

# Exercise 4 - KNN smoothing for income

In [7]:
class KNNIncomeAdder(BaseEstimator, TransformerMixin):
    def __init__(self, n_neighbors=5):
        self.n_neighbors = n_neighbors
    def fit(self, X, y):
        self.knn_ = KNeighborsRegressor(n_neighbors=self.n_neighbors)
        self.knn_.fit(X[["latitude","longitude"]], y)
        return self
    def transform(self, X):
        return self.knn_.predict(X[["latitude","longitude"]]).reshape(-1,1)

preprocessing_knn = ColumnTransformer([("knn_income", KNNIncomeAdder(5), ["latitude","longitude"]),
                                       ("num", num_pipeline, num_attribs),("cat", OneHotEncoder(handle_unknown="ignore"), cat_attribs)])
svr_knn = make_pipeline(preprocessing_knn, SVR(kernel="linear", C=100))
svr_knn.fit(X_train_small,y_train_small)
pred = svr_knn.predict(X_train_small)
print("Training RMSE:", np.sqrt(mean_squared_error(y_train_small,pred)))

Training RMSE: 3035208.27580265


Here, we use latitude + longitude → find nearby districts with KNN to predict their median income then add that prediction as a new feature which can be considered as a smoothing parameter and it performs worse than other models.

# Exercise 5 - RandomisedSearchCV for the entire Pipeline

In [8]:
full_pipeline = Pipeline([("preprocessing", preprocessing_knn),("svr", SVR())])
param_distribs = {"preprocessing__knn_income__n_neighbors": randint(2,30),"svr__kernel":["linear","rbf"],"svr__C":uniform(1,100),
                  "svr__gamma":uniform(0.01,1)}

search = RandomizedSearchCV(full_pipeline,param_distributions=param_distribs,n_iter=3,cv=3,scoring="neg_root_mean_squared_error",
                            n_jobs=-1,random_state=42)
search.fit(X_train_small,y_train_small)
print("Best parameters:",search.best_params_)
print("Best CV RMSE:",-search.best_score_)

Best parameters: {'preprocessing__knn_income__n_neighbors': 22, 'svr__C': np.float64(16.60186404424365), 'svr__gamma': np.float64(0.16599452033620266), 'svr__kernel': 'linear'}
Best CV RMSE: 101973.12778561759


# Exercise 6 - Custom StandardScaler

In [10]:
class StandardScalerClone:
    def fit(self, X):
        X_array = np.asarray(X)
        self.n_features_in_ = X_array.shape[1]
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = np.array(X.columns)
        self.mean_ = np.mean(X_array, axis=0)
        self.scale_ = np.std(X_array, axis=0)
        return self

    def transform(self, X):
        X_array = np.asarray(X)
        return (X_array - self.mean_) / self.scale_

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def inverse_transform(self, X):
        return np.asarray(X) * self.scale_ + self.mean_

    def get_feature_names_out(self, input_features=None):
        if input_features is not None:
            input_features = np.asarray(input_features)
            if len(input_features) != self.n_features_in_:
                raise ValueError("Wrong number of features")
            if hasattr(self, "feature_names_in_") and not np.array_equal(input_features, self.feature_names_in_):
                raise ValueError("Feature names do not match")
            return input_features
        if hasattr(self, "feature_names_in_"):
            return self.feature_names_in_
        return np.array([f"x{i}" for i in range(self.n_features_in_)])

X = pd.DataFrame({"age":[10,20,30],"income":[100,200,300]})

scaler = StandardScalerClone()
X_scaled = scaler.fit_transform(X)
X_back = scaler.inverse_transform(X_scaled)

print("Scaled:")
print(X_scaled)

print("\nRecovered:")
print(X_back)

print("\nFeature names:")
print(scaler.feature_names_in_)

print("\nOutput names:")
print(scaler.get_feature_names_out())

X = np.array([[1,2],[3,4],[5,6]])

scaler = StandardScalerClone()
scaler.fit(X)

print(scaler.get_feature_names_out())

Scaled:
[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]

Recovered:
[[ 10. 100.]
 [ 20. 200.]
 [ 30. 300.]]

Feature names:
['age' 'income']

Output names:
['age' 'income']
['x0' 'x1']


The custom scaler now reproduces the main functionality of StandardScaler, including scaling, inverse transformation, and feature-name handling.